In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

In [20]:
import os

SAVE_DIR = '/content/drive/MyDrive/2025-2 통계계산특론/우리코드/00_HSData/CNN_macro'

In [11]:
import random

def set_seed(val):
    torch.manual_seed(val)
    torch.cuda.manual_seed(val)
    # torch.cuda.manual_seed_all(val)  # 멀티 GPU를 사용하는 경우
    np.random.seed(val)
    random.seed(val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## **DATA LOAD**

In [12]:
FILE_PATH = '/content/drive/MyDrive/2025-2 통계계산특론/우리코드/00_ourData/macroeco_data_lag1.csv'  # lag1
WINDOW_SIZE = 6                     # 6개월치 패턴을 한 번에 봄
output_dims = [5, 10, 20, 50]

# data load
macro_lag1 = pd.read_csv(FILE_PATH)
df1 = macro_lag1[macro_lag1["sasdate"]<= '2022-03-01']

In [13]:
df1.head()

,sasdate,RPI,W875RX1,DPCERA3M086SBEA,CMRMTSPLx,RETAILx,INDPRO,IPFPNSS,IPFINAL,IPCONGD,...,DNDGRG3M086SBEA,DSERRG3M086SBEA,CES0600000008,CES2000000008,CES3000000008,UMCSENTx,DTCOLNVHFNM,DTCTHFNM,INVEST,VIXCLSx
0,1995-12-01,9396.526,8094.0,56.158,857932.126,204032.0,73.0305,82.3776,80.4264,92.5508,...,69.665,58.318,13.09,14.77,12.49,91.0,81073.0,285844.04,893.1924,12.2880
1,1996-01-01,9428.309,8102.2,55.905,849368.258,203793.0,72.6125,81.5501,79.5142,91.4690,...,70.054,58.393,13.23,15.00,12.60,89.3,82026.0,285496.58,890.7512,15.0690
2,1996-02-01,9506.777,8172.2,56.334,857042.936,206875.0,73.6752,82.8225,80.9324,92.8050,...,70.214,58.501,13.18,14.91,12.56,88.5,84076.0,286101.17,898.9080,16.5672
3,1996-03-01,9539.993,8196.0,56.607,857469.507,207650.0,73.5850,82.6687,80.5749,92.2432,...,70.645,58.634,13.15,14.93,12.50,93.7,84340.0,287684.78,895.5430,18.7075
4,1996-04-01,9565.104,8210.4,56.816,865141.781,209294.0,74.3172,83.5322,81.6807,93.0933,...,71.093,58.815,13.30,14.98,12.69,92.7,86461.0,289724.92,896.9737,16.7505


In [ ]:
df1.tail()

,sasdate,RPI,W875RX1,DPCERA3M086SBEA,CMRMTSPLx,RETAILx,INDPRO,IPFPNSS,IPFINAL,IPCONGD,...,DNDGRG3M086SBEA,DSERRG3M086SBEA,CES0600000008,CES2000000008,CES3000000008,UMCSENTx,DTCOLNVHFNM,DTCTHFNM,INVEST,VIXCLSx
311,2021-11-01,19227.878,15657.2,113.734,1470618.0,624874.0,100.8654,99.3175,98.7320,101.4207,...,109.334,113.013,27.00,31.06,24.22,67.4,447955.69,937886.33,5632.1544,19.1586
312,2021-12-01,19170.498,15610.8,113.307,1467940.0,619938.0,100.5726,98.9529,98.2933,100.5504,...,110.124,113.599,27.16,31.21,24.38,70.6,448582.66,934525.50,5693.4714,21.2985
313,2022-01-01,19062.795,15496.6,113.372,1485162.0,631509.0,100.1856,98.6614,98.0805,100.8153,...,110.841,114.049,27.29,31.37,24.52,67.2,448109.20,929241.11,5785.8072,22.9143
314,2022-02-01,19079.074,15483.6,113.448,1472992.0,638101.0,100.8064,99.2788,98.5635,101.0018,...,112.353,114.490,27.41,31.55,24.58,62.8,452636.47,928906.99,5824.9792,26.1429
315,2022-03-01,18998.629,15420.6,114.041,1470559.0,651027.0,101.3907,99.5817,99.0481,101.4487,...,115.074,115.153,27.52,31.66,24.74,59.4,446825.24,918241.74,5846.7078,26.9368


In [14]:
df1['sasdate'] = pd.to_datetime(df1['sasdate'])
df1['year'] = df1['sasdate'].dt.year #split용 연도 변수 추가

/tmp/ipython-input-2612622762.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['sasdate'] = pd.to_datetime(df1['sasdate'])
/tmp/ipython-input-2612622762.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['year'] = df1['sasdate'].dt.year #split용 연도 변수 추가


In [15]:
feature_cols = [c for c in df1.columns if c not in ['sasdate', 'year']]
print(f"전체 데이터 기간: {df1['year'].min()} ~ {df1['year'].max()}")

전체 데이터 기간: 1995 ~ 2022


# **Model**

In [16]:
class MultiHeadCNNEncoder(nn.Module):
    def __init__(self, input_dim, output_dims=[5, 10, 20, 50]):
        super(MultiHeadCNNEncoder, self).__init__()
        self.output_dims = output_dims
        self.heads = nn.ModuleList()
        for out_dim in output_dims:
            head = nn.Sequential(
                nn.Conv1d(input_dim, out_dim * 2, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_dim * 2),
                nn.LeakyReLU(0.1),
                nn.Conv1d(out_dim * 2, out_dim, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_dim),
                nn.LeakyReLU(0.1),
                nn.AdaptiveMaxPool1d(1),
                nn.Flatten()
            )
            self.heads.append(head)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        results = {}
        for i, head in enumerate(self.heads):
            results[f'hidden_{self.output_dims[i]}'] = head(x)
        return results

# 윈도우 생성 함수
def create_sliding_window(data, dates, window):
    X, D = [], []
    for i in range(len(data) - window + 1):
        X.append(data[i:i+window])
        D.append(dates[i+window-1])  # 마지막 날짜를 window의 대표 날짜로
    return np.array(X), np.array(D)

# **Split**

In [17]:
split_year = [
    (2014, 2015, 2017, 2018),
    (2015, 2016, 2018, 2019),
    (2016, 2017, 2019, 2020),
    (2017, 2018, 2020, 2021)
]

In [21]:
for i, (train_end, valid_start, valid_end, test_year) in enumerate(split_year):
    print(f"\n======== Round {i+1}: Test Year {test_year} ========")
    print(f"Train: 1997 ~ {train_end}")
    print(f"Valid: {valid_start} ~ {valid_end}")
    print(f"Test : {test_year}")

    train_start = pd.Timestamp(1997,1,1) - pd.DateOffset(months=WINDOW_SIZE-1)

    # train/valid/test split
    train_df = df1[(df1['sasdate'] >= train_start) &
                   (df1['sasdate'] <= pd.Timestamp(train_end, 12, 31))].copy()

    valid_start_adj = pd.Timestamp(valid_start, 1, 1) - pd.DateOffset(months=WINDOW_SIZE-1)
    valid_df = df1[(df1['sasdate'] >= valid_start_adj) &
                   (df1['sasdate'] <= pd.Timestamp(valid_end, 12, 31))].copy()

    test_start = pd.Timestamp(test_year,1,1) - pd.DateOffset(months=WINDOW_SIZE-1)
    test_end = pd.Timestamp(test_year,12,31)
    test_df = df1[(df1['sasdate'] >= test_start) &
                  (df1['sasdate'] <= test_end)].copy()

    # 스케일링
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols])

    train_scaled = scaler.transform(train_df[feature_cols])
    valid_scaled = scaler.transform(valid_df[feature_cols])
    test_scaled  = scaler.transform(test_df[feature_cols])

    # window
    X_train, dates_train = create_sliding_window(
        train_scaled, train_df['sasdate'].values, WINDOW_SIZE
    )

    X_valid, dates_valid = create_sliding_window(
        valid_scaled, valid_df['sasdate'].values, WINDOW_SIZE
    )

    X_test, dates_test = create_sliding_window(
        test_scaled, test_df['sasdate'].values, WINDOW_SIZE
    )

    # 텐서 변환
    tensors = {
        'train': torch.FloatTensor(X_train),
        'valid': torch.FloatTensor(X_valid),
        'test':  torch.FloatTensor(X_test)
    }
    dates_dict = {'train': dates_train, 'valid': dates_valid, 'test': dates_test}

    # 모델 실행 및 저장
    # 매년 초기화해서 특징 추출 (랜덤 가중치)
    model = MultiHeadCNNEncoder(len(feature_cols))
    model.eval()

    # 저장할 서브 폴더
    round_dir = os.path.join(SAVE_DIR, f'Test_{test_year}')
    os.makedirs(round_dir, exist_ok=True)

    with torch.no_grad():
        for split in ['train', 'valid', 'test']:
            features = model(tensors[split])

            for dim in output_dims:
                feats = features[f'hidden_{dim}'].numpy()
                cols = [f'hs_{dim}_{k}' for k in range(dim)]

                temp_df = pd.DataFrame(feats, columns=cols)
                temp_df.insert(0, 'sasdate', dates_dict[split])

                file_name = f'hidden_states_{split}_{dim}.csv'
                save_path = os.path.join(round_dir, file_name)

                temp_df.to_csv(save_path, index=False)

    print(f"  -> {test_year}년도 데이터 저장 완료")

print("\n 완료")


======== Round 1: Test Year 2018 ========
Train: 1997 ~ 2014
Valid: 2015 ~ 2017
Test : 2018
  -> 2018년도 데이터 저장 완료

======== Round 2: Test Year 2019 ========
Train: 1997 ~ 2015
Valid: 2016 ~ 2018
Test : 2019
  -> 2019년도 데이터 저장 완료

======== Round 3: Test Year 2020 ========
Train: 1997 ~ 2016
Valid: 2017 ~ 2019
Test : 2020
  -> 2020년도 데이터 저장 완료

======== Round 4: Test Year 2021 ========
Train: 1997 ~ 2017
Valid: 2018 ~ 2020
Test : 2021
  -> 2021년도 데이터 저장 완료

 완료
